# Task 3B — Gradient Boost (HistGradientBoosting) for MAR Suitability

Converted from `Task_3B_Gradient boost.py` on 2025-12-29 09:05:25.

Run cells **top-to-bottom**. Outputs are printed/previews are shown for each major step.

## 1) Library imports

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import rasterio

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.ensemble import HistGradientBoostingClassifier

print("Imports OK.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


Imports OK.
numpy: 1.26.4
pandas: 2.3.3
geopandas: 1.1.2
rasterio: 1.4.4


## 2) User inputs (Edit this cell)


In [2]:
# -------------------- USER INPUTS --------------------
DATASET = r"E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv"
REF_RASTER = r"E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif"

OUT_DIR = r"E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON"
OUT_RASTER_DIR = None  # leave None to auto-create OUT_DIR\rasters

FEATURES = ["AET", "LULC", "P", "RZSM", "TEMP", "SOIL"]
TARGET = "MAR_suitable"
GROUP_COL = "pixel_id"

TRAIN_FRAC = 0.70
RANDOM_STATE = 42
NEG_POS_RATIO = 5

# HGB hyperparameters
MAX_ITER = 400
LEARNING_RATE = 0.05
MAX_LEAF_NODES = 31
MAX_DEPTH = None
MIN_SAMPLES_LEAF = 20
L2_REG = 0.0

# Classification thresholds
PROB_THRESHOLD = 0.50
LOW_TH = 0.33
HIGH_TH = 0.66

PERSIST_THRESHOLD = 0.70

OUT_METRICS    = None
OUT_FEATIMP    = None
OUT_PRED_TABLE = None
OUT_COMMON_CSV = None
OUT_COMMON_SHP = None

print("DATASET:", DATASET)
print("REF_RASTER:", REF_RASTER)
print("OUT_DIR:", OUT_DIR)
print("FEATURES:", FEATURES)


DATASET: E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv
REF_RASTER: E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif
OUT_DIR: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON
FEATURES: ['AET', 'LULC', 'P', 'RZSM', 'TEMP', 'SOIL']


## 3) Create output folders + check inputs

In [3]:
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

if OUT_RASTER_DIR is None:
    OUT_RASTER_DIR = os.path.join(OUT_DIR, "rasters")

ensure_dir(OUT_DIR)
ensure_dir(OUT_RASTER_DIR)

OUT_METRICS    = OUT_METRICS    or os.path.join(OUT_DIR, "hgb_metrics.csv")
OUT_FEATIMP    = OUT_FEATIMP    or os.path.join(OUT_DIR, "hgb_feature_importance_perm.csv")
OUT_PRED_TABLE = OUT_PRED_TABLE or os.path.join(OUT_DIR, "hgb_predictions_all_records.csv")
OUT_COMMON_CSV = OUT_COMMON_CSV or os.path.join(OUT_DIR, "hgb_common_new_suitable_pixels.csv")
OUT_COMMON_SHP = OUT_COMMON_SHP or os.path.join(OUT_DIR, "hgb_common_new_suitable_pixels.shp")

missing = False
for p, name in [(DATASET,"DATASET"), (REF_RASTER,"REF_RASTER")]:
    if not os.path.exists(p):
        print("❌ Missing:", name, "->", p)
        missing = True
    else:
        print("✅ Found:", name)

print("✅ Output folder:", OUT_DIR)
print("✅ Raster folder :", OUT_RASTER_DIR)
print("Outputs:")
print("  OUT_METRICS   :", OUT_METRICS)
print("  OUT_FEATIMP   :", OUT_FEATIMP)
print("  OUT_PRED_TABLE:", OUT_PRED_TABLE)
print("  OUT_COMMON_CSV:", OUT_COMMON_CSV)
print("  OUT_COMMON_SHP:", OUT_COMMON_SHP)

if missing:
    raise FileNotFoundError("Fix missing paths in Config and rerun.")


✅ Found: DATASET
✅ Found: REF_RASTER
✅ Output folder: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON
✅ Raster folder : E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\rasters
Outputs:
  OUT_METRICS   : E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_metrics.csv
  OUT_FEATIMP   : E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_feature_importance_perm.csv
  OUT_PRED_TABLE: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_predictions_all_records.csv
  OUT_COMMON_CSV: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_common_new_suitable_pixels.csv
  OUT_COMMON_SHP: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_common_new_suitable_pixels.shp


## 4) Helper functions

In [4]:
def export_points_shp(df_in, out_shp):
    """Export lon/lat points to shapefile (EPSG:4326) with attributes."""
    if df_in.empty:
        print(f"WARNING: Empty output; not writing: {out_shp}")
        return
    gdf = gpd.GeoDataFrame(
        df_in.copy(),
        geometry=[Point(xy) for xy in zip(df_in["lon"], df_in["lat"])],
        crs="EPSG:4326"
    )
    gdf.to_file(out_shp)
    print("Saved:", out_shp)

def prob_to_lmh_cols(prob_series, low=0.33, high=0.66):
    """Vectorized LMH class code + label from probability."""
    codes = np.full(prob_series.shape, np.nan, dtype="float32")
    labels = np.full(prob_series.shape, None, dtype=object)

    p = prob_series.to_numpy()

    m0 = np.isnan(p)
    m1 = (~m0) & (p < low)
    m2 = (~m0) & (p >= low) & (p < high)
    m3 = (~m0) & (p >= high)

    codes[m1] = 1; labels[m1] = "Low"
    codes[m2] = 2; labels[m2] = "Medium"
    codes[m3] = 3; labels[m3] = "High"

    return codes, labels

def permutation_importance_auc(model, X_val, y_val, random_state=42, n_repeats=3):
    """
    Simple permutation importance based on drop in ROC-AUC.
    Returns dataframe with mean/std importance over repeats.
    """
    rng = np.random.RandomState(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_auc = roc_auc_score(y_val, base_prob)

    importances = {c: [] for c in X_val.columns}
    Xp = X_val.copy()

    for _ in range(n_repeats):
        for col in X_val.columns:
            saved = Xp[col].to_numpy().copy()
            rng.shuffle(Xp[col].values)
            prob = model.predict_proba(Xp)[:, 1]
            auc = roc_auc_score(y_val, prob)
            importances[col].append(base_auc - auc)
            Xp[col] = saved

    rows = []
    for col, vals in importances.items():
        rows.append({
            "feature": col,
            "importance_perm_auc_mean": float(np.mean(vals)),
            "importance_perm_auc_std": float(np.std(vals)),
        })
    return pd.DataFrame(rows).sort_values("importance_perm_auc_mean", ascending=False)

print("Helpers loaded.")


Helpers loaded.


## 5) Load + clean dataset

In [5]:
df = pd.read_csv(DATASET)

required = set(["year", "pixel_id", "lon", "lat"] + FEATURES + [TARGET])
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in dataset: {missing_cols}")

df = df.dropna(subset=FEATURES + [TARGET, "year", "pixel_id", "lon", "lat"]).copy()
df["year"] = df["year"].astype(int)
df[TARGET] = df[TARGET].astype(int)

years = sorted(df["year"].unique())
n_years = len(years)

print("Years:", years, "| N years:", n_years)
print("All records:", len(df))
print("Overall class counts:\n", df[TARGET].value_counts())
display(df.head(5))


Years: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024] | N years: 11
All records: 513414
Overall class counts:
 MAR_suitable
0    511566
1      1848
Name: count, dtype: int64


,year,pixel_id,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_suitable
0,2014,383,27.65,70.05,206.53160,30.0,233.09563,0.270511,-0.002954,4.0,0
1,2014,384,27.75,70.05,201.74350,30.0,229.78244,0.293638,0.038191,4.0,0
2,2014,385,27.85,70.05,213.14160,30.0,229.21902,0.272055,0.080452,4.0,0
3,2014,386,27.95,70.05,228.95428,30.0,230.52446,0.347064,0.099918,4.0,0
4,2014,826,26.65,69.95,247.35196,30.0,262.63400,0.258783,-0.309810,4.0,0


## 6) Negative sampling per year (NEG_POS_RATIO × positives)

In [6]:
train_parts = []
for y in years:
    dyy = df[df["year"] == y].copy()
    pos = dyy[dyy[TARGET] == 1]
    neg = dyy[dyy[TARGET] == 0]

    n_pos = len(pos)
    if n_pos == 0:
        continue

    n_neg_need = min(len(neg), NEG_POS_RATIO * n_pos)
    if n_neg_need > 0:
        neg_sample = neg.sample(n=n_neg_need, random_state=RANDOM_STATE)
        train_parts.append(pos)
        train_parts.append(neg_sample)
    else:
        train_parts.append(pos)

train_df = pd.concat(train_parts, ignore_index=True)

print("Training dataset after negative sampling:")
print("Records:", len(train_df))
print("Class counts:\n", train_df[TARGET].value_counts())
display(train_df.head(5))


Training dataset after negative sampling:
Records: 11088
Class counts:
 MAR_suitable
0    9240
1    1848
Name: count, dtype: int64


,year,pixel_id,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_suitable
0,2014,32999,27.65,62.85,565.99664,111.0,716.23865,0.310972,5.217053,4.0,1
1,2014,34304,22.25,62.55,499.93250,111.0,625.83234,0.304080,5.714527,4.0,1
2,2014,35698,25.75,62.25,485.78363,111.0,650.00620,0.318120,5.517731,4.0,1
3,2014,38431,27.25,61.65,477.28930,111.0,698.70950,0.315008,5.782207,4.0,1
4,2014,39300,23.55,61.45,544.11880,80.0,665.37850,0.377936,6.207617,4.0,1


## 7) Train/test split (grouped by pixel_id)

In [7]:
X = train_df[FEATURES]
y = train_df[TARGET]
groups = train_df[GROUP_COL]

gss = GroupShuffleSplit(n_splits=1, train_size=TRAIN_FRAC, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train class counts:\n", y_train.value_counts())
print("Test class counts:\n", y_test.value_counts())


Train rows: 7755 | Test rows: 3333
Train class counts:
 MAR_suitable
0    6468
1    1287
Name: count, dtype: int64
Test class counts:
 MAR_suitable
0    2772
1     561
Name: count, dtype: int64


## 8) Train HistGradientBoosting + evaluate

In [8]:
# If your sklearn errors on class_weight, comment that line and rerun.
hgb = HistGradientBoostingClassifier(
    loss="log_loss",
    learning_rate=LEARNING_RATE,
    max_iter=MAX_ITER,
    max_leaf_nodes=MAX_LEAF_NODES,
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    l2_regularization=L2_REG,
    random_state=RANDOM_STATE,
    class_weight="balanced"
)

hgb.fit(X_train, y_train)

y_pred = hgb.predict(X_test)
y_prob = hgb.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
roc = roc_auc_score(y_test, y_prob)

print("\n=== TEST METRICS ===")
print(f"Accuracy   : {acc:.3f}")
print(f"Precision  : {prec:.3f}")
print(f"Recall     : {rec:.3f}")
print(f"F1-score   : {f1:.3f}")
print(f"ROC-AUC    : {roc:.3f}")
print("\nConfusion Matrix [ [TN FP] [FN TP] ]:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(
    y_test, y_pred,
    target_names=["Unsuitable(0)", "Suitable(1)"],
    digits=3,
    zero_division=0
))

pd.DataFrame([{
    "Accuracy": acc,
    "Precision": prec,
    "Recall": rec,
    "F1_score": f1,
    "ROC_AUC": roc,
    "Train_frac": TRAIN_FRAC,
    "NEG_POS_RATIO": NEG_POS_RATIO,
    "model": "HistGradientBoostingClassifier",
    "max_iter": MAX_ITER,
    "learning_rate": LEARNING_RATE,
    "max_leaf_nodes": MAX_LEAF_NODES,
    "max_depth": MAX_DEPTH,
    "min_samples_leaf": MIN_SAMPLES_LEAF,
    "l2_regularization": L2_REG,
    "prob_threshold": PROB_THRESHOLD,
    "persist_threshold": PERSIST_THRESHOLD,
    "random_state": RANDOM_STATE
}]).to_csv(OUT_METRICS, index=False)

fi = permutation_importance_auc(hgb, X_test, y_test, random_state=RANDOM_STATE, n_repeats=3)
fi.to_csv(OUT_FEATIMP, index=False)

print("\nSaved metrics:", OUT_METRICS)
print("Saved permutation feature importance:", OUT_FEATIMP)
display(fi)



=== TEST METRICS ===
Accuracy   : 0.821
Precision  : 0.476
Recall     : 0.595
F1-score   : 0.529
ROC-AUC    : 0.827

Confusion Matrix [ [TN FP] [FN TP] ]:
 [[2404  368]
 [ 227  334]]

Classification Report:
                precision    recall  f1-score   support

Unsuitable(0)      0.914     0.867     0.890      2772
  Suitable(1)      0.476     0.595     0.529       561

     accuracy                          0.821      3333
    macro avg      0.695     0.731     0.709      3333
 weighted avg      0.840     0.821     0.829      3333


Saved metrics: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_metrics.csv
Saved permutation feature importance: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_feature_importance_perm.csv


,feature,importance_perm_auc_mean,importance_perm_auc_std
4,TEMP,0.159950,0.005332
1,LULC,0.108833,0.002182
3,RZSM,0.094891,0.004083
0,AET,0.079616,0.001524
2,P,0.029007,0.002718
5,SOIL,0.012810,0.004553


## 9) Predict for all pixels (all years) + LMH reclass

In [9]:
df["MAR_probability"] = hgb.predict_proba(df[FEATURES])[:, 1]
df["MAR_predicted"] = (df["MAR_probability"] >= PROB_THRESHOLD).astype(int)

df["suit_class_code"], df["suit_class"] = prob_to_lmh_cols(df["MAR_probability"], LOW_TH, HIGH_TH)

OUT_PRED_TABLE = OUT_PRED_TABLE or os.path.join(OUT_DIR, "hgb_predictions_all_records.csv")
df.to_csv(OUT_PRED_TABLE, index=False)

print("Saved prediction table:", OUT_PRED_TABLE)
display(df[["year","pixel_id","MAR_probability","MAR_predicted","suit_class_code","suit_class"]].head(10))


Saved prediction table: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_predictions_all_records.csv


,year,pixel_id,MAR_probability,MAR_predicted,suit_class_code,suit_class
0,2014,383,0.000002,0,1.0,Low
1,2014,384,0.000002,0,1.0,Low
2,2014,385,0.000002,0,1.0,Low
3,2014,386,0.000003,0,1.0,Low
4,2014,826,0.000002,0,1.0,Low
5,2014,828,0.000002,0,1.0,Low
6,2014,833,0.000002,0,1.0,Low
7,2014,834,0.000002,0,1.0,Low
8,2014,835,0.000004,0,1.0,Low
9,2014,836,0.000003,0,1.0,Low


## 10) Common/persistent new suitable pixels (CSV + SHP)
Sites appearing more than 70% (8 years) of time suitable in past 11 years

In [11]:
new_all = df[(df[TARGET] == 0) & (df["MAR_predicted"] == 1)].copy()

if new_all.empty:
    print("WARNING: No new suitable records found. Common output will be empty.")
    pd.DataFrame().to_csv(OUT_COMMON_CSV, index=False)
else:
    common_summary = (
        new_all.groupby("pixel_id")
              .agg(
                  years_new_suitable=("MAR_predicted", "sum"),
                  mean_probability=("MAR_probability", "mean"),
                  lon=("lon", "first"),
                  lat=("lat", "first"),
                  AET=("AET", "mean"),
                  P=("P", "mean"),
                  RZSM=("RZSM", "mean"),
                  TEMP=("TEMP", "mean"),
                  LULC=("LULC", "first"),
                  SOIL=("SOIL", "first"),
              )
              .reset_index()
    )
    common_summary["years_total"] = n_years
    common_summary["new_suitable_ratio"] = common_summary["years_new_suitable"] / n_years

    common = common_summary[common_summary["new_suitable_ratio"] >= PERSIST_THRESHOLD].copy()
    common = common.sort_values(["new_suitable_ratio", "mean_probability"], ascending=False)

    common["suit_class_code"], common["suit_class"] = prob_to_lmh_cols(common["mean_probability"], LOW_TH, HIGH_TH)

    common.to_csv(OUT_COMMON_CSV, index=False)
    print("Saved common new suitable CSV:", OUT_COMMON_CSV)
    print("Common new suitable pixels:", len(common))
    display(common.head(20))

    export_points_shp(common, OUT_COMMON_SHP)


Saved common new suitable CSV: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_common_new_suitable_pixels.csv
Common new suitable pixels: 2524


,pixel_id,years_new_suitable,mean_probability,lon,lat,AET,P,RZSM,TEMP,LULC,SOIL,years_total,new_suitable_ratio,suit_class_code,suit_class
4531,81696,11,0.999088,4.95,52.05,628.518620,890.502617,0.489828,11.374297,40.0,3.0,11,1.0,3.0,High
5507,84879,11,0.998794,6.15,51.35,728.158938,898.673682,0.276931,11.714931,50.0,4.0,11,1.0,3.0,High
5376,84420,11,0.998758,5.55,51.45,659.074230,900.763445,0.267838,11.719770,50.0,4.0,11,1.0,3.0,High
5951,86230,11,0.998705,5.35,51.05,738.516769,905.910252,0.326889,11.691012,50.0,4.0,11,1.0,3.0,High
6103,86684,11,0.998602,5.45,50.95,725.320995,904.867065,0.338971,11.655430,50.0,4.0,11,1.0,3.0,High
5246,83960,11,0.998595,4.85,51.55,687.261009,904.412382,0.246446,11.621044,50.0,4.0,11,1.0,3.0,High
5375,84419,11,0.998575,5.45,51.45,651.288611,907.591747,0.248180,11.703120,50.0,4.0,11,1.0,3.0,High
5377,84421,11,0.998565,5.65,51.45,677.415616,893.607417,0.251546,11.711544,50.0,4.0,11,1.0,3.0,High
5249,83963,11,0.998457,5.15,51.55,683.587785,905.884518,0.253776,11.605342,50.0,4.0,11,1.0,3.0,High
6102,86683,11,0.998255,5.35,50.95,711.846255,905.948883,0.321787,11.700745,50.0,4.0,11,1.0,3.0,High


Saved: E:\VUB\Final\PixelDataFrames\GB_MAR_Outputs_COMMON\hgb_common_new_suitable_pixels.shp


C:\Users\tejue\AppData\Local\Temp\ipykernel_7588\3499321352.py:11: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(out_shp)
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'years_new_suitable' to 'years_new_'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'mean_probability' to 'mean_proba'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'years_total' to 'years_tota'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'new_suitable_ratio' to 'new_suitab'
  ogr_write(
C:\Users\tejue\anaconda3\lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'suit_class_code' to 'suit_class'
  ogr_write(
C:\Users\tej

## 11) Export year-wise probability + LMH rasters

In [12]:
with rasterio.open(REF_RASTER) as ref:
    profile = ref.profile.copy()
    width, height = ref.width, ref.height

df["row"] = (df["pixel_id"].astype("int64") // width).astype("int32")
df["col"] = (df["pixel_id"].astype("int64") % width).astype("int32")

prob_profile = profile.copy()
prob_profile.update(dtype="float32", count=1, nodata=np.nan, compress="lzw")

cls_profile = profile.copy()
cls_profile.update(dtype="uint8", count=1, nodata=0, compress="lzw")

for y in years:
    dyy = df[df["year"] == y]

    prob_arr = np.full((height, width), np.nan, dtype="float32")
    cls_arr = np.zeros((height, width), dtype="uint8")

    rr = dyy["row"].to_numpy()
    cc = dyy["col"].to_numpy()
    pp = dyy["MAR_probability"].to_numpy(dtype="float32")

    m = (rr >= 0) & (rr < height) & (cc >= 0) & (cc < width)
    rr, cc, pp = rr[m], cc[m], pp[m]

    prob_arr[rr, cc] = pp

    valid = ~np.isnan(prob_arr)
    cls_arr[valid & (prob_arr < LOW_TH)] = 1
    cls_arr[valid & (prob_arr >= LOW_TH) & (prob_arr < HIGH_TH)] = 2
    cls_arr[valid & (prob_arr >= HIGH_TH)] = 3

    prob_tif = os.path.join(OUT_RASTER_DIR, f"suitability_probability_{y}.tif")
    cls_tif  = os.path.join(OUT_RASTER_DIR, f"suitability_class_LMH_{y}.tif")

    with rasterio.open(prob_tif, "w", **prob_profile) as dst:
        dst.write(prob_arr, 1)

    with rasterio.open(cls_tif, "w", **cls_profile) as dst:
        dst.write(cls_arr, 1)

    print(f"Saved rasters for {y}: {os.path.basename(prob_tif)} , {os.path.basename(cls_tif)}")

print("DONE. All outputs saved in:", OUT_DIR)


Saved rasters for 2014: suitability_probability_2014.tif , suitability_class_LMH_2014.tif
Saved rasters for 2015: suitability_probability_2015.tif , suitability_class_LMH_2015.tif
Saved rasters for 2016: suitability_probability_2016.tif , suitability_class_LMH_2016.tif
Saved rasters for 2017: suitability_probability_2017.tif , suitability_class_LMH_2017.tif
Saved rasters for 2018: suitability_probability_2018.tif , suitability_class_LMH_2018.tif
Saved rasters for 2019: suitability_probability_2019.tif , suitability_class_LMH_2019.tif
Saved rasters for 2020: suitability_probability_2020.tif , suitability_class_LMH_2020.tif
Saved rasters for 2021: suitability_probability_2021.tif , suitability_class_LMH_2021.tif
Saved rasters for 2022: suitability_probability_2022.tif , suitability_class_LMH_2022.tif
Saved rasters for 2023: suitability_probability_2023.tif , suitability_class_LMH_2023.tif
Saved rasters for 2024: suitability_probability_2024.tif , suitability_class_LMH_2024.tif
DONE. All 